In [9]:
# imstall these required packagess once
# !pip install -q kaggle
# !pip install -q nltk
# !pip install -q tqdm

# Imports
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from google.colab import files
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D,  Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix


print('imports done!')

imports done!


In [10]:
# loading the dataset of facial expressions from kaggle !
import kagglehub

path = kagglehub.dataset_download("msambare/fer2013")

print("Path to dataset files: ", path)
print("Contents of dataset directory:")
print(os.listdir(path))

# image paths for the dataset
train_dir = os.path.join(path, "train")
val_dir = os.path.join(path, "test")

Path to dataset files:  /kaggle/input/fer2013
Contents of dataset directory:
['test', 'train']


In [11]:
# since ResNet expects images of size 224x224 RGB we have to resize & normalize!
# do for training and test dataset

image_size = (224, 224)
batch_size = 32

# do data augmentation to the training set for better generalization
train_datagenerator = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# training set
train_generator = train_datagenerator.flow_from_directory(
    train_dir,
    target_size=image_size,
    batch_size=batch_size,
    color_mode='rgb',
    class_mode='categorical'
)

# validation set
val_generator = train_datagenerator.flow_from_directory(
    val_dir,
    target_size=image_size,
    batch_size=batch_size,
    color_mode='rgb',
    class_mode='categorical'
)


Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


In [15]:
# ResNet Transfer Layer Model! (starting Transfer Learning)
# we chose this because its already pretrained with imagenet - only planning to train the last layer
# faster and more efficient for our project

# well use Adam Optimizer in ResNet 50

# loading ResNet50 w/o last layer then we freeze!
base_res_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_res_model.trainable = False

# should we do avgpooling or maxpooling???? idk yettt

x = base_res_model.output
x = GlobalAveragePooling2D()(x)
# x = MaxPooling2D(pool_size=(7, 7))(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)

predictions = Dense(train_generator.num_classes, activation='softmax')(x)

# build & save the model !!
model = Model(inputs=base_res_model.input, outputs=predictions)

# Compile it
model.compile(optimizer=Adam(learning_rate=1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Summary
model.summary()



Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_3[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [16]:
# doing early stopping so that it stops training when the valdation loss stops improving
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# train the model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stopping]
)

# Save the model to a file
model.save('facial_exp_resnet50_model.keras')  #Will create a directory that has all the related files and saves everything there

files.download("facial_exp_resnet50_model.keras")
saved_model = tf.keras.models.load_model('facial_exp_resnet50_model.keras')
score = saved_model.evaluate(val_generator)

print("Testing Loss:", score[0], "Testing Accuracy:", score[1])
print("Model saved as 'facial_expression_resnet_model'")

Epoch 1/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 471s 513ms/step - accuracy: 0.1630 - loss: 2.3791 - val_accuracy: 0.1780 - val_loss: 1.8295
Epoch 2/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 450s 501ms/step - accuracy: 0.1868 - loss: 2.0995 - val_accuracy: 0.1782 - val_loss: 1.8310
Epoch 3/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 447s 498ms/step - accuracy: 0.1864 - loss: 2.0231 - val_accuracy: 0.1970 - val_loss: 1.8326
Epoch 4/10
898/898 ━━━━━━━━━━━━━━━━━━━━ 442s 492ms/step - accuracy: 0.1931 - loss: 1.9638 - val_accuracy: 0.2473 - val_loss: 1.8355


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

225/225 ━━━━━━━━━━━━━━━━━━━━ 96s 397ms/step - accuracy: 0.1798 - loss: 1.8239
Testing Loss: 1.828909993171692 Testing Accuracy: 0.17887991666793823
Model saved as 'facial_expression_resnet_model'


In [20]:
loss, accuracy = model.evaluate(val_generator)
print(f"✅ Test Loss: {loss:.4f}")
print(f"✅ Test Accuracy: {accuracy:.4f}")

225/225 ━━━━━━━━━━━━━━━━━━━━ 96s 427ms/step - accuracy: 0.1829 - loss: 1.8199
✅ Test Loss: 1.8291
✅ Test Accuracy: 0.1797


In [21]:
# Predict probabilities
y_pred_probs = model.predict(val_generator)
y_pred = np.argmax(y_pred_probs, axis=1)

# True labels
y_true = val_generator.classes

# Class labels
class_labels = list(val_generator.class_indices.keys())

# Confusion matrix
# cm = confusion_matrix(y_true, y_pred)
# plt.figure(figsize=(8,6))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
# plt.title("Confusion Matrix")
# plt.xlabel("Predicted")
# plt.ylabel("True")
# plt.show()

# Classification report
print("Classification Report:\n", classification_report(y_true, y_pred, target_names=class_labels))

225/225 ━━━━━━━━━━━━━━━━━━━━ 95s 403ms/step
Classification Report:
               precision    recall  f1-score   support

       angry       0.00      0.00      0.00       958
     disgust       0.00      0.00      0.00       111
        fear       0.00      0.00      0.00      1024
       happy       0.25      0.14      0.18      1774
     neutral       0.17      0.86      0.29      1233
         sad       0.00      0.00      0.00      1247
    surprise       0.00      0.00      0.00       831

    accuracy                           0.18      7178
   macro avg       0.06      0.14      0.07      7178
weighted avg       0.09      0.18      0.09      7178



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [23]:
from tensorflow.keras.preprocessing import image

img_path = "/content/man.jpg"
img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

pred = model.predict(img_array)
pred_class = class_labels[np.argmax(pred)]

print(f"Predicted Expression: {pred_class}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
Predicted Expression: neutral
